# Quan sat ket qua cac module trong `src/`

Notebook nay minh hoa dau vao, dau ra va cach cac module ket noi voi nhau. De notebook chay nhanh va khong can tai ImageNet weights, model duoc tao voi `pretrained=False`.


In [1]:
from pathlib import Path
import sys

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
    
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_module import CIFAR10DataModule
from src.logger import TensorBoardLogger
from src.model_builder import TransferModelBuilder
from src.trainer import ModelTrainer

torch.manual_seed(42)
print("Project root:", ROOT)
print("PyTorch:", torch.__version__)
print("Device:", "cuda:0" if torch.cuda.is_available() else "cpu")

Project root: D:\UTH\Deeplearning\lab\Practice-2---Hands-on-practice-with-pre-trained-neural-network-architectures
PyTorch: 2.14.1+cpu
Device: cpu


In [3]:
print("Cuda có sẵn không:", torch.cuda.is_available())

Cuda có sẵn không: False


In [4]:
print(torch.__version__)

2.14.1+cpu


## 1. `data_module.py`: tao dataset va DataLoader


In [2]:
data_module = CIFAR10DataModule(
    data_dir=str(ROOT / "data"),
    batch_size=4,
    num_workers=0,
    image_size=224,
)
data_module.setup()

print("Train dataset:", type(data_module.train_dataset).__name__)
print("Train samples:", len(data_module.train_dataset))
print("Test samples:", len(data_module.test_dataset))
print("Train transform:", data_module.train_transform)
print("Test transform:", data_module.test_transform)

train_images, train_labels = next(iter(data_module.train_dataloader()))
print("Train image batch shape:", train_images.shape)
print("Train label batch shape:", train_labels.shape)
print("Train image dtype:", train_images.dtype)
print("First labels:", train_labels.tolist())

Train dataset: CIFAR10
Train samples: 50000
Test samples: 10000
Train transform: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
Test transform: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
Train image batch shape: torch.Size([4, 3, 224, 224])
Train label batch shape: torch.Size([4])
Train image dtype: torch.float32
First labels: [6, 0, 4, 1]


## 2. `model_builder.py`: tao model va freeze backbone


In [3]:
builder = TransferModelBuilder(
    model_name="resnet18",
    num_classes=10,
    pretrained=True,
)
model = builder.build()
total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(model)
print("Output classes:", model.fc.out_features)
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Frozen parameters:", total_params - trainable_params)

with torch.no_grad():
    model_output = model(train_images)
print("Model output shape:", model_output.shape)
print("Predicted classes:", model_output.argmax(dim=1).tolist())

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

## 3. `trainer.py`: mot buoc train va evaluate

De quan sat nhanh, phan nay dung mot tap du lieu gia lap nho co cung shape voi anh CIFAR-10 sau resize.

In [6]:
demo_images = torch.randn(8, 3, 224, 224)
demo_labels = torch.randint(0, 10, (8,))
demo_loader = DataLoader(
    TensorDataset(demo_images, demo_labels),
    batch_size=4,
    shuffle=False,
)

demo_model = TransferModelBuilder(
    model_name="resnet18",
    num_classes=10,
    pretrained=True,
).build()
trainer = ModelTrainer(
    model=demo_model,
    train_dataloader=demo_loader,
    test_dataloader=demo_loader,
    learning_rate=0.001,
    optimizer_name="Adam",
    device=torch.device("cpu"),
    weight_decay=0.0001,
)

train_loss, train_accuracy = trainer.training_loop()
test_loss, test_accuracy = trainer.evaluate()
print(f"Train loss: {train_loss:.4f}")
print(f"Train accuracy: {train_accuracy:.2f}%")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2f}%")

Train loss: 2.2155
Train accuracy: 37.50%
Test loss: 2.1271
Test accuracy: 25.00%


## 4. `logger.py`: ghi TensorBoard va checkpoint


In [7]:
logger = TensorBoardLogger(
    log_dir=ROOT / "runs",
    experiment_name="src_output_demo",
    checkpoint_dir=ROOT / "checkpoints",
)
logger.add_scalar("Demo/train_loss", train_loss, step=1)
logger.add_scalar("Demo/test_accuracy", test_accuracy, step=1)
logger.add_image_grid(
    "Demo/images",
    train_images,
    nrow=2,
    step=0,
    unnormalize=True,
)
checkpoint_path = logger.save_checkpoint(
    model=demo_model,
    optimizer=trainer.optimizer,
    epoch=1,
    metrics={"test_accuracy": test_accuracy},
    filename="src_output_demo.pth",
)
logger.flush()
logger.close()

print("TensorBoard directory:", ROOT / "runs" / "src_output_demo")
print("Checkpoint path:", checkpoint_path)

TensorBoard directory: D:\UTH\Deeplearning\lab\Practice-2---Hands-on-practice-with-pre-trained-neural-network-architectures\runs\src_output_demo
Checkpoint path: D:\UTH\Deeplearning\lab\Practice-2---Hands-on-practice-with-pre-trained-neural-network-architectures\checkpoints\src_output_demo.pth


Mo TensorBoard bang lenh:

```powershell
tensorboard --logdir=runs
```

Sau do mo `http://localhost:6006` de xem cac scalar va image da ghi.